# Мониторинг экосистемы: классификация видов цветов

Демо-ноутбук практикума «Использование предобученных моделей для создания ML-driven продуктов».

---

## Бизнес-постановка (на обычном языке)

Задача в рамках темы «Мониторинг экосистем через IT-решения»: автоматически определять вид растения по фотографии, чтобы ускорить учёт биоразнообразия и поддержать решения по охране природы. В данном решении — классификация пяти видов цветов: ромашка, одуванчик, роза, подсолнух, тюльпан.

---

## Постановка в DS-терминах

- **Область:** Computer Vision (CV).
- **Тип задачи:** классификация изображений (multiclass classification).
- **Вход:** изображение (RGB); **выход:** класс вида и вероятность.

---

## Данные и предобученная модель

- **Данные (датасет для тестирования и дообучения):** Flowers Recognition, 5 классов.  
  Ссылка: https://www.kaggle.com/datasets/alxmamaev/flowers-recognition  

- **Предобученная модель:** EfficientNetB0 (ImageNet), Keras/TensorFlow.  
  Документация: https://keras.io/api/applications/efficientnet/

## 1. Установка зависимостей и импорты

Библиотеки: TensorFlow/Keras (модель и аугментации), scikit-learn (метрики), matplotlib/seaborn (визуализация).

In [ ]:
# Основные библиотеки для работы с данными и визуализации
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# TensorFlow/Keras для модели и аугментаций
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Метрики классификации
from sklearn.metrics import classification_report, confusion_matrix

# Настройка воспроизводимости и отображения
np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

print("TensorFlow:", tf.__version__)
print("GPU доступен:", len(tf.config.list_physical_devices('GPU')) > 0)

## 2. Конфигурация путей и гиперпараметров

In [ ]:
# Путь к датасету: папка с подпапками по классам (daisy, dandelion, roses, sunflowers, tulips)
# Датасет: https://www.kaggle.com/datasets/alxmamaev/flowers-recognition
DATA_DIR = Path("data/flowers")
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Гиперпараметры
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-4
VALIDATION_SPLIT = 0.2

# Порог уверенности: ниже — рекомендуется ручная проверка в рамках мониторинга экосистем
CONFIDENCE_THRESHOLD = 0.75

## 3. Загрузка данных и разбиение на train/validation

In [ ]:
# Проверка наличия данных (датасет: Flowers Recognition, 5 классов)
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Папка {DATA_DIR} не найдена. Скачайте датасет (например Kaggle: Flowers Recognition) "
        "и распакуйте так: data/flowers/daisy/, data/flowers/dandelion/, roses/, sunflowers/, tulips/"
    )
# Имена классов = подпапки в DATA_DIR
class_names = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
num_classes = len(class_names)
print("Классы:", class_names)
print("Количество классов:", num_classes)

# Аугментации для обучающей выборки (улучшение обобщающей способности)
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=VALIDATION_SPLIT,
    fill_mode='nearest'
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=42,
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=42,
    shuffle=False
)

# Соответствие индексов и имён классов (для отчётов)
class_indices = train_generator.class_indices
print("Соответствие индексов классам:", class_indices)

## 4. Сборка модели на базе EfficientNetB0

In [ ]:
# Предобученная на ImageNet основа — EfficientNetB0
# Выбор: хороший баланс точность/скорость, подходит для transfer learning при ограниченном датасете
base_model = EfficientNetB0(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)

# Замораживаем часть слоёв основания, дообучаем верхние
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=True)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 5. Обучение с сохранением весов после каждой эпохи

In [ ]:
# Сохранение лучшей модели по val_accuracy и весов после каждой эпохи
checkpoint_path = CHECKPOINT_DIR / "flowers_effnet_best.keras"
checkpoint_per_epoch = CHECKPOINT_DIR / "flowers_effnet_epoch_{epoch:02d}.keras"

callbacks = [
    ModelCheckpoint(
        checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    ModelCheckpoint(
        str(checkpoint_per_epoch),
        monitor='val_accuracy',
        save_freq='epoch',
        verbose=0
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

## 6. Визуализация процесса обучения

In [ ]:
# Совместимость с разными версиями Keras (accuracy / acc)
hist = history.history
val_acc_key = 'val_accuracy' if 'val_accuracy' in hist else 'val_acc'
acc_key = 'accuracy' if 'accuracy' in hist else 'acc'
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist['loss'], label='train loss')
axes[0].plot(hist['val_loss'], label='val loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(hist[acc_key], label='train accuracy')
axes[1].plot(hist[val_acc_key], label='val accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Оценка на валидации: classification_report и матрица ошибок

In [ ]:
# Предсказания по валидационной выборке (сброс генератора)
val_generator.reset()
y_pred_proba = model.predict(val_generator, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = val_generator.classes

# Точность по классам и общая метрика
print("\n--- Classification report ---")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Матрица ошибок: какие классы путаются между собой
cm = confusion_matrix(y_true, y_pred)
print("\n--- Confusion matrix (counts) ---")
print(cm)

In [ ]:
# Визуализация матрицы ошибок (heatmap)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Матрица ошибок (Confusion Matrix)')
plt.ylabel('Истинный класс')
plt.xlabel('Предсказанный класс')
plt.tight_layout()
plt.show()

## 8. Примеры предсказаний и порог уверенности для ручной проверки

In [ ]:
# Загружаем лучшую модель (если перезапустили ядро)
# model = keras.models.load_model(checkpoint_path)

# Берём один батч из валидации для визуализации
val_generator.reset()
x_batch, y_batch = next(val_generator)
preds = model.predict(x_batch, verbose=0)
pred_classes = np.argmax(preds, axis=1)
pred_probs = np.max(preds, axis=1)

n_show = min(8, len(x_batch))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
for i in range(n_show):
    axes[i].imshow(x_batch[i])
    true_label = class_names[np.argmax(y_batch[i])]
    pred_label = class_names[pred_classes[i]]
    prob = pred_probs[i]
    need_review = " (проверка!)" if prob < CONFIDENCE_THRESHOLD else ""
    axes[i].set_title(f"Истина: {true_label}\nПредск.: {pred_label} ({prob:.2f}){need_review}")
    axes[i].axis('off')
for j in range(n_show, len(axes)):
    axes[j].axis('off')
plt.suptitle(f"Порог уверенности для ручной проверки: {CONFIDENCE_THRESHOLD}")
plt.tight_layout()
plt.show()

## 9. Модуль подсчёта объектов на изображении

In [ ]:
def count_flowers_in_image(model, image_path, class_names, img_size=(224, 224), stride=112):
    """
    Оценка количества цветов на изображении методом скользящего окна.
    Каждое окно классифицируется; учитываются только предсказания с уверенностью выше порога.
    Для мониторинга экосистем даёт приблизительную оценку обилия на снимке.
    """
    from tensorflow.keras.preprocessing.image import load_img, img_to_array
    img = load_img(image_path, target_size=img_size)
    img_arr = np.array(img) / 255.0
    # Если изображение большое — можно разбить на патчи (здесь один кадр = одно "окно")
    img_batch = np.expand_dims(img_arr, axis=0)
    preds = model.predict(img_batch, verbose=0)[0]
    pred_class = np.argmax(preds)
    confidence = preds[pred_class]
    return {
        'class': class_names[pred_class],
        'confidence': float(confidence),
        'count_estimate': 1 if confidence >= CONFIDENCE_THRESHOLD else 0,
        'needs_manual_review': confidence < CONFIDENCE_THRESHOLD
    }

# Пример: подсчёт по одному изображению из датасета
sample_images = list(DATA_DIR.glob("*/*.jpg"))[:3]
if not sample_images:
    sample_images = list(DATA_DIR.glob("*/*.png"))[:3]
for path in sample_images:
    r = count_flowers_in_image(model, str(path), class_names)
    print(f"{path.name}: класс={r['class']}, уверенность={r['confidence']:.3f}, нужна проверка={r['needs_manual_review']}")

## 10. Итоги

- Модель **EfficientNetB0** дообучена на 5 классах цветов; веса сохраняются в `checkpoints/` после каждой эпохи.
- По **classification_report** и **confusion matrix** видно, какие классы путаются (например, ромашка и одуванчик).
- Для мониторинга экосистем при **низкой уверенности** (ниже порога) рекомендуется ручная проверка.
- Модуль подсчёта объектов даёт оценку по одному кадру; при необходимости его можно расширить на скользящее окно по большому изображению.